# Connecting with Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Change working directory
%cd '/content/drive/MyDrive/Colab Notebooks/deep_acelerometry'

/content/drive/MyDrive/Colab Notebooks/deep_acelerometry


# Setup

In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, ReLU, Add, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, Flatten, Concatenate
from tensorflow.keras.regularizers import l2
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
# Check that we are using GPU
tf.config.experimental.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

# Dataset

### 1. Load target labels

In [5]:
target = pd.read_csv('target.csv')

In [6]:
target

,SEQN,DXXNK_TSCORE,CLASSES,TARGET_BINARY
0,73557.0,NaN,NaN,NaN
1,73558.0,-0.358333,normal,0.0
2,73559.0,NaN,NaN,NaN
3,73561.0,-1.133333,osteopenia,1.0
4,73562.0,0.316667,normal,0.0
...,...,...,...,...
3703,83721.0,-0.266667,normal,0.0
3704,83723.0,1.191667,normal,0.0
3705,83724.0,NaN,NaN,NaN
3706,83726.0,0.841667,normal,0.0


### 2. Load relevant features for classification: gender, age and body mass.

In [7]:
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')

In [8]:
X_train.shape, X_test.shape

((2003, 9), (501, 9))

In [9]:
X_train.iloc[:,:4]

,SEQN,RIAGENDR,RIDAGEYR,BMXWT
0,76988.0,1.0,67.0,68.3
1,82083.0,2.0,57.0,67.4
2,75435.0,2.0,63.0,60.3
3,76741.0,2.0,68.0,70.1
4,78834.0,2.0,56.0,57.1
...,...,...,...,...
1998,74345.0,1.0,56.0,79.2
1999,82126.0,1.0,56.0,70.5
2000,78475.0,2.0,77.0,71.7
2001,74862.0,2.0,48.0,66.1


In [10]:
# Combine them into a single DataFrame
features_df = pd.concat([X_train.iloc[:,:4], X_test.iloc[:,:4]]).reset_index(drop=True)

In [11]:
features_df

,SEQN,RIAGENDR,RIDAGEYR,BMXWT
0,76988.0,1.0,67.0,68.3
1,82083.0,2.0,57.0,67.4
2,75435.0,2.0,63.0,60.3
3,76741.0,2.0,68.0,70.1
4,78834.0,2.0,56.0,57.1
...,...,...,...,...
2499,79610.0,2.0,66.0,74.3
2500,77717.0,1.0,48.0,131.7
2501,75946.0,2.0,71.0,92.4
2502,75081.0,2.0,66.0,64.7


### 3. Physical activity data (tensors with GAF images)

Choose sampling frequency, which determines image size

In [12]:
# directory = '/content/drive/MyDrive/Colab Notebooks/deep_acelerometry/tensors_288' # 1 sample every 5 minutes
directory = '/content/drive/MyDrive/Colab Notebooks/deep_acelerometry/tensors_144' # 1 sample every 10 minutes
image_size = int(directory.split('_')[2])

# Read tensor filenames
all_files = [f for f in os.listdir(directory) if f.endswith('.npy')]

# Split tensor files according to the original 80/20 split
train_files = [f for f in all_files if f.startswith('train_')]
test_files = [f for f in all_files if f.startswith('test_')]

print("Total tensor files:", len(all_files))
print("Training tensor files:", len(train_files))
print("Test tensor files:", len(test_files))


Total tensor files: 2504
Training tensor files: 2003
Test tensor files: 501


In [13]:
image_size

144

# Data generator

### Step 1: Define the Generator Class
You'll subclass tf.keras.utils.Sequence to ensure your generator handles batch processing and shuffling properly.

In [26]:
class DataGenerator(Sequence):
    def __init__(self, directory, file_list, target_df, features_df, batch_size=32, shuffle=True, **kwargs):
        super().__init__(**kwargs)

        self.directory = directory
        self.file_list = file_list
        self.target_df = target_df
        self.features_df = features_df
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.file_list))

        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __len__(self):
        return int(np.ceil(len(self.file_list) / self.batch_size))

    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_files = [self.file_list[k] for k in indexes]
        return self.__load_data(batch_files)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __load_data(self, batch_files):
        """Loads data for a batch."""
        x = []
        y = []
        features = []

        for file_name in batch_files:
            file_path = os.path.join(self.directory, file_name)
            data = np.load(file_path)
            x.append(data)

            subject_id = float(file_name.split('_')[1])

            # Retrieve target label
            target_label = self.target_df[
                self.target_df['SEQN'] == subject_id
            ]['TARGET_BINARY'].iloc[0]
            y.append(target_label)

            # Retrieve additional features
            feature_row = self.features_df[
                self.features_df['SEQN'] == subject_id
            ][['RIAGENDR', 'RIDAGEYR', 'BMXWT']]

            if not feature_row.empty:
                features.append(feature_row.iloc[0].values)
            else:
                features.append(np.array([np.nan, np.nan, np.nan]))

        # Important for newer Keras versions:
        # multi-input models should receive a tuple, not a list.
        # Everything is cast to float32 to match TensorFlow/Keras tensors.
        return (
            (np.array(x, dtype=np.float32), np.array(features, dtype=np.float32)),
            np.array(y, dtype=np.float32)
        )

### Step 2: Instantiate the Data Generator
Now you can create instances of DataGenerator for both training and validation.

In [27]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
batch_size = 32

def extract_seqn_from_filename(file_name):
    """
    Extract participant SEQN from filenames such as:
    train_12345_144.npy or test_12345_144.npy
    """
    return float(file_name.split('_')[1])

# Get TARGET_BINARY labels for the original training files.
# These labels are used only to create a stratified internal validation split.
train_labels = [
    target.loc[target['SEQN'] == extract_seqn_from_filename(f), 'TARGET_BINARY'].iloc[0]
    for f in train_files
]

# Split the original training set into:
# - train_inner_files: used to fit the CNN
# - val_files: used for early stopping/model selection
# The original test_files remain untouched until final evaluation.
train_inner_files, val_files = train_test_split(
    train_files,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=train_labels
)

print("Internal training files:", len(train_inner_files))
print("Internal validation files:", len(val_files))
print("Final untouched test files:", len(test_files))

# Create generators
train_generator = DataGenerator(
    directory=directory,
    file_list=train_inner_files,
    target_df=target,
    features_df=features_df,
    batch_size=batch_size,
    shuffle=True
)

val_generator = DataGenerator(
    directory=directory,
    file_list=val_files,
    target_df=target,
    features_df=features_df,
    batch_size=batch_size,
    shuffle=False
)

test_generator = DataGenerator(
    directory=directory,
    file_list=test_files,
    target_df=target,
    features_df=features_df,
    batch_size=batch_size,
    shuffle=False
)

Internal training files: 1602
Internal validation files: 401
Final untouched test files: 501


train_generator → used for model fitting

val_generator   → used for early stopping

test_generator  → used only once, after training

# Auxiliary functions

In [22]:
from tensorflow.keras import backend as K

In [28]:
def recall_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    recall = true_positives / (possible_positives + K.epsilon())
    return recall

def precision_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    return precision

def f1_m(y_true, y_pred):
    precision = precision_m(y_true, y_pred)
    recall = recall_m(y_true, y_pred)
    return 2*((precision*recall)/(precision+recall+K.epsilon()))

In [29]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

def get_true_labels(generator):
    """
    Extract true labels from a Keras Sequence generator.
    """
    y_true = []

    for i in range(len(generator)):
        _, y_batch = generator[i]
        y_true.extend(y_batch)

    return np.array(y_true).astype(int)


def evaluate_classification_model(model, test_generator, model_name):
    """
    Evaluate a binary classification model on the untouched test set.
    Positive class = osteopenia/osteoporosis.
    Negative class = normal bone status.
    """
    y_true = get_true_labels(test_generator)

    y_prob = model.predict(test_generator).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Sensitivity": tp / (tp + fn),   # Recall for osteopenia/osteoporosis
        "Specificity": tn / (tn + fp),   # True negative rate for normal status
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    predictions = pd.DataFrame({
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred
    })

    return results, predictions

# Compile and train

**Modelo 1)** 5 veces Conv2D + BatchNormalization + MaxPooling [ONLY EXPLORATORY. NOT REPORTED IN PAPER]

In [ ]:
# Define the input layer for the images
image_input = Input(shape=(image_size, image_size, 7))
# Define the input layer for the additional features
features_input = Input(shape=(3,))

# CNN layers - making the network deeper and more complex
nof_filters = 64  # Increased the number of filters
x = Conv2D(nof_filters, (3, 3), activation='relu', padding='same')(image_input)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(nof_filters * 2, (3, 3), activation='relu', padding='same')(x)  # Second Conv layer
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(nof_filters * 4, (3, 3), activation='relu', padding='same')(x)  # Third Conv layer
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(nof_filters * 8, (3, 3), activation='relu', padding='same')(x)  # Fourth Conv layer
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(nof_filters * 16, (3, 3), activation='relu', padding='same')(x)  # Fifth Conv layer
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Dropout(0.1)(x)  # Increased dropout rate after more layers
x = Flatten()(x)

# Concatenate CNN output and additional features
x = Concatenate()([x, features_input])

# Dense layers
nof_neurons = 64  # Increased neurons
x = Dense(nof_neurons, activation='relu')(x)
x = Dropout(0.3)(x)  # Increased dropout rate in dense layer
x = BatchNormalization()(x)
output = Dense(1, activation='sigmoid')(x)

# Create the model
model = Model(inputs=[image_input, features_input], outputs=output)
#model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', recall_m, f1_m])

# Add callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(train_generator, validation_data=test_generator, epochs=20, callbacks=[early_stopping])


Epoch 1/20
62/62 [==============================] - 26s 340ms/step - loss: 0.8159 - accuracy: 0.5398 - recall_m: 0.4459 - f1_m: 0.4204 - val_loss: 2.6895 - val_accuracy: 0.3896 - val_recall_m: 1.0000 - val_f1_m: 0.5585
Epoch 2/20
62/62 [==============================] - 20s 321ms/step - loss: 0.7072 - accuracy: 0.6058 - recall_m: 0.4029 - f1_m: 0.4340 - val_loss: 0.6567 - val_accuracy: 0.6292 - val_recall_m: 0.2273 - val_f1_m: 0.3049
Epoch 3/20
62/62 [==============================] - 19s 310ms/step - loss: 0.7188 - accuracy: 0.6003 - recall_m: 0.3738 - f1_m: 0.4113 - val_loss: 0.6734 - val_accuracy: 0.6042 - val_recall_m: 0.4240 - val_f1_m: 0.4518
Epoch 4/20
62/62 [==============================] - 21s 335ms/step - loss: 0.7017 - accuracy: 0.5932 - recall_m: 0.3443 - f1_m: 0.3882 - val_loss: 0.9838 - val_accuracy: 0.5958 - val_recall_m: 0.0048 - val_f1_m: 0.0089
Epoch 5/20
62/62 [==============================] - 19s 307ms/step - loss: 0.6693 - accuracy: 0.6114 - recall_m: 0.3608 - f1

**Modelo 2)** ResNet
- Intenté en lugar ResNet preentrenado para ImageNet (transfer learning), pero tiene una serie de restricciones que lo complican, como el hecho de que admite sólo 3 canales, y no 7.

In [30]:
def residual_block(x, filters, kernel_size=3, stride=1, downsample=False):
    shortcut = x
    if downsample:
        shortcut = Conv2D(filters, 1, strides=stride, padding='same')(x)
        shortcut = BatchNormalization()(shortcut)

    x = Conv2D(filters, kernel_size, padding='same', strides=stride)(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv2D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)

    x = Add()([x, shortcut])
    x = ReLU()(x)
    return x

def tfm_resnet(input_shape, additional_features_shape, num_classes):
    image_input = Input(shape=input_shape)
    features_input = Input(shape=additional_features_shape)

    # Initial convolution
    x = Conv2D(64, (7, 7), strides=2, padding='same')(image_input)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)

    # ResNet blocks
    x = residual_block(x, 64)
    x = residual_block(x, 64)
    x = residual_block(x, 128, stride=2, downsample=True)  # downsample
    x = residual_block(x, 128)
    x = residual_block(x, 256, stride=2, downsample=True)  # downsample
    x = residual_block(x, 256)

    # Global Pooling and output layer for image features
    x = GlobalAveragePooling2D()(x)

    # Concatenate CNN output and additional features
    x = Concatenate()([x, features_input])

    # Dense layers and final classification layer
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = BatchNormalization()(x)
    output = Dense(num_classes, activation='sigmoid')(x)

    # Create model
    model = Model(inputs=[image_input, features_input], outputs=output)
    return model

# Set up and compile the ResNet model
resnet_model = tfm_resnet(
    input_shape=(image_size, image_size, 7),
    additional_features_shape=(3,),
    num_classes=1
)

resnet_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', recall_m, f1_m]
)

# Train the model using the internal validation set, not the final test set
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

resnet_history = resnet_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stopping]
)

# Final evaluation on the untouched test set
resnet_results, resnet_predictions = evaluate_classification_model(
    resnet_model,
    test_generator,
    "ResNet"
)

resnet_results


Epoch 1/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 600s 11s/step - accuracy: 0.6298 - f1_m: 13.2054 - loss: 0.6749 - recall_m: 14.8431 - val_accuracy: 0.6135 - val_f1_m: 0.1429 - val_loss: 93.1372 - val_recall_m: 0.0769
Epoch 2/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 14s 283ms/step - accuracy: 0.6785 - f1_m: 12.1545 - loss: 0.6146 - recall_m: 12.4118 - val_accuracy: 0.6135 - val_f1_m: 0.4065 - val_loss: 2.8500 - val_recall_m: 0.2308
Epoch 3/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 14s 267ms/step - accuracy: 0.6879 - f1_m: 11.2854 - loss: 0.6241 - recall_m: 10.8431 - val_accuracy: 0.6110 - val_f1_m: 0.0000e+00 - val_loss: 8.5580 - val_recall_m: 0.0000e+00
Epoch 4/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 14s 272ms/step - accuracy: 0.6954 - f1_m: 10.8019 - loss: 0.6056 - recall_m: 10.0392 - val_accuracy: 0.6758 - val_f1_m: 5.6178 - val_loss: 0.6366 - val_recall_m: 3.8462
Epoch 5/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 14s 271ms/step - accuracy: 0.7135 - f1_m: 11.2177 - loss: 0.5915 - recall_m: 10.6471 - val_accuracy: 0.3865 - val_f1_m: 16.914

{'Model': 'ResNet',
 'Accuracy': 0.6626746506986028,
 'Sensitivity': np.float64(0.2153846153846154),
 'Specificity': np.float64(0.9477124183006536),
 'Precision': 0.7241379310344828,
 'Recall': 0.2153846153846154,
 'F1-score': 0.33201581027667987,
 'ROC-AUC': np.float64(0.765694653929948),
 'TN': np.int64(290),
 'FP': np.int64(16),
 'FN': np.int64(153),
 'TP': np.int64(42)}

**Modelo 3:** VGG

In [31]:
# Define the input layer for the images
image_input = Input(shape=(image_size, image_size, 7), name='image_input')
# Define the input layer for the additional features
features_input = Input(shape=(3,), name='features_input')

# Block 1
x = Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv1')(image_input)
x = Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv2')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block1_pool')(x)

# Block 2
x = Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv1')(x)
x = Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv2')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block2_pool')(x)

# Block 3
x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv1')(x)
x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv2')(x)
x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv3')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block3_pool')(x)

# Block 4
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv1')(x)
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv2')(x)
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv3')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block4_pool')(x)

# Flatten and concatenate additional features
x = Flatten(name='flatten')(x)
x = Concatenate()([x, features_input])

# Dense Classification Block
x = Dense(4096, activation='relu', name='fc1')(x)
x = Dropout(0.5)(x)  # Incorporating dropout for regularization
x = Dense(4096, activation='relu', name='fc2')(x)
x = Dropout(0.5)(x)  # Incorporating dropout for regularization
output = Dense(1, activation='sigmoid', name='predictions')(x)  # Adjust the output units and activation for binary classification

# Create the VGG model
vgg_model = Model(inputs=[image_input, features_input], outputs=output)

vgg_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', recall_m, f1_m]
)

# Add callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Model summary to check architecture
vgg_model.summary()

# Train the model using the internal validation set, not the final test set
vgg_history = vgg_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stopping]
)

# Final evaluation on the untouched test set
vgg_results, vgg_predictions = evaluate_classification_model(
    vgg_model,
    test_generator,
    "VGG"
)

vgg_results


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 144, 144,  │          0 │ -                 │
│ (InputLayer)        │ 7)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 144, 144,  │      4,096 │ image_input[0][0] │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 144, 144,  │     36,928 │ block1_conv1[0][… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_pool         │ (None, 72, 72,    │          0 │ block1_conv2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv1        │ (None, 72, 72,    │     73,856 │ block1_pool[0][0] │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv2        │ (None, 72, 72,    │    147,584 │ block2_conv1[0][… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 36, 36,    │          0 │ block2_conv2[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv1        │ (None, 36, 36,    │    295,168 │ block2_pool[0][0] │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv2        │ (None, 36, 36,    │    590,080 │ block3_conv1[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv3        │ (None, 36, 36,    │    590,080 │ block3_conv2[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_pool         │ (None, 18, 18,    │          0 │ block3_conv3[0][… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv1        │ (None, 18, 18,    │  1,180,160 │ block3_pool[0][0] │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv2        │ (None, 18, 18,    │  2,359,808 │ block4_conv1[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv3        │ (None, 18, 18,    │  2,359,808 │ block4_conv2[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_pool         │ (None, 9, 9, 512) │          0 │ block4_conv3[0][… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 41472)     │          0 │ block4_pool[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ features_input      │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                 

 Total params: 194,308,673 (741.23 MB)

 Trainable params: 194,308,673 (741.23 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 73s 807ms/step - accuracy: 0.6442 - f1_m: 9.6362 - loss: 1.8992 - recall_m: 10.9020 - val_accuracy: 0.7007 - val_f1_m: 6.3697 - val_loss: 0.6005 - val_recall_m: 4.6154
Epoch 2/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 19s 378ms/step - accuracy: 0.7029 - f1_m: 10.1718 - loss: 0.5932 - recall_m: 9.4510 - val_accuracy: 0.7082 - val_f1_m: 10.8733 - val_loss: 0.5661 - val_recall_m: 10.3846
Epoch 3/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 15s 285ms/step - accuracy: 0.6879 - f1_m: 10.2719 - loss: 0.6011 - recall_m: 9.7647 - val_accuracy: 0.7107 - val_f1_m: 8.0096 - val_loss: 0.5703 - val_recall_m: 6.3077
Epoch 4/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 15s 292ms/step - accuracy: 0.6916 - f1_m: 10.2681 - loss: 0.5832 - recall_m: 9.6667 - val_accuracy: 0.7157 - val_f1_m: 7.9047 - val_loss: 0.5724 - val_recall_m: 6.1538
Epoch 5/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 15s 290ms/step - accuracy: 0.7085 - f1_m: 10.4428 - loss: 0.5860 - recall_m: 9.8235 - val_accuracy: 0.7157 - val_f1_m: 7.9047 - val_loss

{'Model': 'VGG',
 'Accuracy': 0.7125748502994012,
 'Sensitivity': np.float64(0.6153846153846154),
 'Specificity': np.float64(0.7745098039215687),
 'Precision': 0.6349206349206349,
 'Recall': 0.6153846153846154,
 'F1-score': 0.625,
 'ROC-AUC': np.float64(0.7797888386123681),
 'TN': np.int64(237),
 'FP': np.int64(69),
 'FN': np.int64(75),
 'TP': np.int64(120)}

In [32]:
classification_results = pd.DataFrame([
    resnet_results,
    vgg_results
])

classification_results

,Model,Accuracy,Sensitivity,Specificity,Precision,Recall,F1-score,ROC-AUC,TN,FP,FN,TP
0,ResNet,0.662675,0.215385,0.947712,0.724138,0.215385,0.332016,0.765695,290,16,153,42
1,VGG,0.712575,0.615385,0.774510,0.634921,0.615385,0.625000,0.779789,237,69,75,120


In [33]:
classification_results.to_csv("cnn_classification_test_metrics.csv", index=False)

resnet_predictions.to_csv("resnet_classification_test_predictions.csv", index=False)
vgg_predictions.to_csv("vgg_classification_test_predictions.csv", index=False)

resnet_model.save("resnet_classification_final.keras")
vgg_model.save("vgg_classification_final.keras")